# ETL DimFecha

Carga incremental de las fechas de venta desde Stage hacia `dbo.DimFecha`.

In [ ]:
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from pathlib import Path

In [ ]:
# Cargar variables de entorno desde etl/.env.

ruta_env = Path("../.env").resolve()
load_dotenv(dotenv_path=ruta_env)


# Conexión a Stage.
cadena_stage = quote_plus(
    f"DRIVER={{{os.getenv('SQLSERVER_DRIVER')}}};"
    f"SERVER={os.getenv('SQLSERVER_SERVIDOR')};"
    f"DATABASE={os.getenv('SQLSERVER_BASE_DATOS')};"
    f"UID={os.getenv('SQLSERVER_USUARIO')};"
    f"PWD={os.getenv('SQLSERVER_PASSWORD')};"
    "TrustServerCertificate=yes;"
)

engine_stage = create_engine(
    f"mssql+pyodbc:///?odbc_connect={cadena_stage}"
)

# Conexión al Data Warehouse.
cadena_dw = quote_plus(
    f"DRIVER={{{os.getenv('SQLSERVER_DRIVER_DW')}}};"
    f"SERVER={os.getenv('SQLSERVER_SERVIDOR_DW')};"
    f"DATABASE={os.getenv('SQLSERVER_BASE_DATOS_DW')};"
    f"UID={os.getenv('SQLSERVER_USUARIO_DW')};"
    f"PWD={os.getenv('SQLSERVER_PASSWORD_DW')};"
    "TrustServerCertificate=yes;"
)

engine_dw = create_engine(
    f"mssql+pyodbc:///?odbc_connect={cadena_dw}"
)

In [3]:
# Extraer las fechas de las ventas desde Stage.
esquema_stage = os.getenv("SQLSERVER_ESQUEMA")

consulta_fechas = f"SELECT fecha_venta FROM {esquema_stage}.venta"

df_fechas = pd.read_sql(
    consulta_fechas,
    engine_stage
)

registros_extraidos = len(df_fechas)
print("Registros extraídos:", len(df_fechas))

Registros extraídos: 10000


In [4]:
# Descartar fechas nulas y eliminar la hora para conservar un registro por día.
df_fechas["fecha"] = pd.to_datetime(
    df_fechas["fecha_venta"],
    errors="coerce"
).dt.normalize()
df_fechas = df_fechas.dropna(subset=["fecha"])
df_fechas = df_fechas.drop_duplicates(subset=["fecha"]).copy()

nombres_meses = {
    1: "enero",
    2: "febrero",
    3: "marzo",
    4: "abril",
    5: "mayo",
    6: "junio",
    7: "julio",
    8: "agosto",
    9: "septiembre",
    10: "octubre",
    11: "noviembre",
    12: "diciembre"
}

df_fechas["fecha_sk"] = df_fechas["fecha"].dt.strftime("%Y%m%d").astype("int64")
df_fechas["dia"] = df_fechas["fecha"].dt.day
df_fechas["mes"] = df_fechas["fecha"].dt.month
df_fechas["nombre_mes"] = df_fechas["mes"].map(nombres_meses)
df_fechas["trimestre"] = df_fechas["fecha"].dt.quarter
df_fechas["anio"] = df_fechas["fecha"].dt.year
df_fechas["es_fin_semana"] = df_fechas["fecha"].dt.dayofweek >= 5

df_fechas = df_fechas[
    [
        "fecha_sk",
        "fecha",
        "dia",
        "mes",
        "nombre_mes",
        "trimestre",
        "anio",
        "es_fin_semana"
    ]
]

print("Fechas únicas transformadas:", len(df_fechas))

Fechas únicas transformadas: 1336


In [5]:
# Leer las claves de fecha que ya existen en el Data Warehouse.
esquema_dw = os.getenv("SQLSERVER_ESQUEMA_DW")

df_fechas_existentes = pd.read_sql(
    f"SELECT fecha_sk FROM {esquema_dw}.DimFecha",
    engine_dw
)

print("Registros existentes:", len(df_fechas_existentes))

# Conservar únicamente las fechas que todavía no están cargadas.
df_fechas_nuevas = df_fechas[
    ~df_fechas["fecha_sk"].isin(df_fechas_existentes["fecha_sk"])
].copy()

print("Registros nuevos:", len(df_fechas_nuevas))

InterfaceError: (pyodbc.InterfaceError) ('28000', '[28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Login failed for user \'etl\'. (18456) (SQLDriverConnect); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot open database "dw_tienda_friki" requested by the login. The login failed. (4060); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Login failed for user \'etl\'. (18456); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot open database "dw_tienda_friki" requested by the login. The login failed. (4060)')
(Background on this error at: https://sqlalche.me/e/20/rvf5)

In [ ]:
# Insertar las fechas nuevas sin reemplazar los datos existentes.
registros_insertados = 0

if not df_fechas_nuevas.empty:
    df_fechas_nuevas.to_sql(
        name="DimFecha",
        con=engine_dw,
        schema=esquema_dw,
        if_exists="append",
        index=False
    )

    registros_insertados = len(df_fechas_nuevas)

print("Registros extraídos:", registros_extraidos)
print("Registros existentes:", len(df_fechas_existentes))
print("Registros nuevos:", len(df_fechas_nuevas))
print("Registros insertados:", registros_insertados)

# Cerrar conexiones.
engine_stage.dispose()
engine_dw.dispose()